# MNIST 手寫數字辨識訓練

流程跟 Stanley 的 `emnist_train_colab.ipynb` 一樣：資料 -> 模型架構 -> 訓練/評估 -> 量化成 int8 tflite -> 分析 ops -> 轉成 C array。

差異：MNIST 只有數字 0-9（10 類），比 EMNIST 的字母（26 類）單純，適合當第一個「自己訓練+部署」的練習。

跑法：Runtime -> Run all，全部跑完後最後一格會自動下載 `mnist_model_data.h` / `mnist_model_data.cc`。

## 0. 環境:Colab / Kaggle 兩邊都能跑

### TF 版本這件事的真相

文件說線上轉檔器支援到 TF 2.14.1,但 **2.14 只有 cp39-cp311 的 wheel**:

| 平台 | Python | 能裝的最低 TF | 結論 |
|---|---|---|---|
| Colab | 3.13 | 2.19 (cp313 起) | 裝不了 2.14 |
| Kaggle | 3.12 | 2.16 (cp312 起) | 裝不了 2.14 |

所以「釘 TF 2.14.1」在這兩個平台上都做不到。**但真正會咬人的不是版本號,是 Keras 3。**

TF >= 2.16 預設用 Keras 3,`model.save("x.h5")` 存出來的 HDF5 內部結構跟
Keras 2 不一樣(layer config、`model_config` 的 JSON schema 都改過),
Realtek 轉檔器是 Keras 2 時代的東西,讀 Keras 3 的 h5 很可能直接掛。

**解法:裝 `tf-keras` + 設 `TF_USE_LEGACY_KERAS=1`。**
這樣 `tf.keras` 會轉回 Keras 2 的實作,`.h5` 就是舊格式,不需要降 Python。
下面那格的 `USE_LEGACY_KERAS` 就是幹這件事。

### Kaggle 用法

1. kaggle.com -> 右上 `+ Create` -> `New Notebook`
2. 右側 `Session options`:`Internet` 打開(不開不能 pip install)。
   `Accelerator` 找不到的話是手機沒驗證(頭像 -> Settings -> Phone verification);
   這個模型很小,純 CPU 也就 10-20 分鐘,可以先跳過
3. `File -> Import Notebook` -> 上傳這個 .ipynb
4. 把下面那格的 `USE_LEGACY_KERAS` 改成 True,跑那格,裝完 `Run -> Restart session`,
   然後從第 1 格 Run All(**env var 必須在 import tensorflow 之前設好,所以一定要 restart**)
5. 下載產物:右側 `Output` 面板點檔名;或 `Save Version -> Save & Run All`,
   在該 version 的 Output 頁一次抓全部


In [ ]:
import os, sys, shutil, subprocess

IS_KAGGLE = os.path.exists("/kaggle/working")
OUT_DIR = "/kaggle/working" if IS_KAGGLE else "."

# --- Keras 2 相容模式 ---------------------------------------------------
# TF >= 2.16 預設 Keras 3,存出的 .h5 內部 schema 跟 Keras 2 不同,
# Realtek 線上轉檔器很可能讀不懂。改成 True -> 裝 tf-keras 並切回 Keras 2。
# 注意:TF_USE_LEGACY_KERAS 必須在 import tensorflow 之前設好,
#      所以裝完一定要 restart session,再從第 1 格重跑。
USE_LEGACY_KERAS = False

if USE_LEGACY_KERAS:
    os.environ["TF_USE_LEGACY_KERAS"] = "1"
    try:
        import tf_keras  # noqa: F401
        print("tf-keras 已就緒,TF_USE_LEGACY_KERAS=1")
    except ImportError:
        print("裝 tf-keras ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tf-keras"], check=False)
        print("裝完了 -> 現在去 Run/Runtime 選單 Restart session,再從第 1 格 Run All")

print("env:", "Kaggle" if IS_KAGGLE else "Colab/local",
      "| python", sys.version.split()[0],
      "| legacy_keras", os.environ.get("TF_USE_LEGACY_KERAS", "0"),
      "| OUT_DIR", OUT_DIR)


def save_output(path):
    # 把產物放到可下載的位置:Kaggle 靠 Output 面板,Colab 直接彈下載
    dst = os.path.join(OUT_DIR, os.path.basename(path))
    if os.path.abspath(dst) != os.path.abspath(path):
        shutil.copy(path, dst)
    if not IS_KAGGLE:
        try:
            import importlib
            importlib.import_module("google.colab.files").download(dst)
            return dst
        except Exception as e:
            print("不是 Colab,檔案在 " + dst + " (" + str(e) + ")")
            return dst
    print("Kaggle: 右邊 Output 面板下載 -> " + dst)
    return dst


In [ ]:
import os
import numpy as np
import tensorflow as tf

print("tensorflow:", tf.__version__)

# tf.keras 到底接到誰?tf_keras (Keras 2) 的 shim 沒有 keras.__version__,
# 所以不能直接讀,要看實作模組的名字。
_impl = getattr(tf.keras, "__name__", "?")
if _impl == "?" or "keras" not in _impl:
    _impl = type(tf.keras.layers.Dense).__module__
try:
    import tf_keras as _k
    _kv = _k.__version__
except ImportError:
    import keras as _k
    _kv = _k.__version__

IS_KERAS2 = _impl.startswith("tf_keras") or _kv.startswith("2")
print("tf.keras impl:", _impl)
print("keras version:", _kv, "->", "Keras 2 (轉檔器要的)" if IS_KERAS2 else "Keras 3")
print("TF_USE_LEGACY_KERAS =", os.environ.get("TF_USE_LEGACY_KERAS", "(unset)"))

if not IS_KERAS2:
    print("!! 現在是 Keras 3,存出的 .h5 Realtek 轉檔器可能讀不懂。")
    print("!! 要修:回第 0 節把 USE_LEGACY_KERAS 改 True -> Restart session -> Run All")


2.20.0


## 1. 資料 -- MNIST

`tf.keras.datasets.mnist` 內建就有，不用像 EMNIST 一樣手動下載。60000 張訓練圖、10000 張測試圖，28x28 灰階。

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print("train:", x_train.shape, y_train.shape)
print("test:", x_test.shape, y_test.shape)

NUM_CLASSES = 10

# Input contract the firmware expects: pixel [0,255] -> float [0,1].
# (Not [-1,1] this time -- the AMB82 online conversion tool only supports
# a pure scale factor (output = input * scale), no mean/offset subtraction,
# so we keep training-time normalization to a plain scale too, for both
# targets. Doesn't affect accuracy, just which equivalent range we use.)
x_train_f = (x_train.astype(np.float32) / 255.0)[..., None]
x_test_f = (x_test.astype(np.float32) / 255.0)[..., None]

train: (60000, 28, 28) (60000,)
test: (10000, 28, 28) (10000,)


## 2. 模型架構

跟 Stanley 的 EMNIST CNN 幾乎一樣，只是最後 Dense 層改成 10 類（0-9）。

In [ ]:
def build_model():
    data_augmentation = tf.keras.Sequential()
    data_augmentation.add(tf.keras.layers.RandomRotation(0.035, input_shape=(28, 28, 1)))
    data_augmentation.add(tf.keras.layers.RandomTranslation(0.08, 0.08))

    model = tf.keras.Sequential()
    model.add(data_augmentation)
    model.add(tf.keras.layers.Conv2D(32, 3, activation="relu"))
    model.add(tf.keras.layers.MaxPooling2D(2))
    model.add(tf.keras.layers.Conv2D(64, 3, activation="relu"))
    model.add(tf.keras.layers.MaxPooling2D(2))
    model.add(tf.keras.layers.Reshape((5 * 5 * 64,)))
    model.add(tf.keras.layers.Dropout(0.5))
    model.add(tf.keras.layers.Dense(NUM_CLASSES))
    model.add(tf.keras.layers.Softmax())
    return model

build_model().summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_5 (Sequential)       │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_3 (Reshape)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │        16,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ softmax_3 (Softmax)             │ (None, 10)             │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 34,826 (136.04 KB)

 Trainable params: 34,826 (136.04 KB)

 Non-trainable params: 0 (0.00 B)

## 3. 訓練、評估

`EPOCHS` 可以自己調，預設 10（MNIST 比 EMNIST 簡單，不用到 25）。

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

tf.random.set_seed(42)
model = build_model()
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(x_train_f, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                     validation_split=0.1, verbose=2)

test_loss, test_acc = model.evaluate(x_test_f, y_test, verbose=0)
FLOAT_ACC = test_acc * 100
print(f"FLOAT test accuracy: {FLOAT_ACC:.2f}%  (loss={test_loss:.4f})")

# Keras 2 的 .keras 原生格式不接受 include_optimizer(Keras 3 會默默吞掉),
# .h5 格式才支援 -> 統一用 .h5,並留 fallback 讓兩種 Keras 都能跑
MODEL_PATH = "mnist_tiny.h5"
try:
    model.save(MODEL_PATH, include_optimizer=False)
except (ValueError, TypeError):
    model.save(MODEL_PATH)
print("saved:", MODEL_PATH)


Epoch 1/10
422/422 - 5s - 12ms/step - accuracy: 0.8018 - loss: 0.6254 - val_accuracy: 0.9748 - val_loss: 0.0940
Epoch 2/10
422/422 - 3s - 7ms/step - accuracy: 0.9249 - loss: 0.2469 - val_accuracy: 0.9830 - val_loss: 0.0629
Epoch 3/10
422/422 - 3s - 8ms/step - accuracy: 0.9418 - loss: 0.1938 - val_accuracy: 0.9853 - val_loss: 0.0558
Epoch 4/10
422/422 - 3s - 6ms/step - accuracy: 0.9506 - loss: 0.1625 - val_accuracy: 0.9860 - val_loss: 0.0478
Epoch 5/10
422/422 - 3s - 6ms/step - accuracy: 0.9565 - loss: 0.1433 - val_accuracy: 0.9872 - val_loss: 0.0420
Epoch 6/10
422/422 - 3s - 7ms/step - accuracy: 0.9591 - loss: 0.1341 - val_accuracy: 0.9885 - val_loss: 0.0395
Epoch 7/10
422/422 - 4s - 8ms/step - accuracy: 0.9631 - loss: 0.1215 - val_accuracy: 0.9905 - val_loss: 0.0355
Epoch 8/10
422/422 - 3s - 6ms/step - accuracy: 0.9661 - loss: 0.1115 - val_accuracy: 0.9908 - val_loss: 0.0342
Epoch 9/10
422/422 - 3s - 6ms/step - accuracy: 0.9679 - loss: 0.1037 - val_accuracy: 0.9907 - val_loss: 0.0338


## 4. 量化成 int8 TF Lite

In [ ]:
SAMPLES = 300
EVAL_N = 2000

def representative_dataset():
    rng = np.random.default_rng(0)
    idx = rng.choice(len(x_train_f), SAMPLES, replace=False)
    for i in idx:
        yield [x_train_f[i:i + 1]]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()

TFLITE_PATH = "mnist_int8.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"Wrote: {TFLITE_PATH}  ({len(tflite_model)} bytes)")

# Quick int8 accuracy check
interp = tf.lite.Interpreter(model_content=tflite_model)
interp.allocate_tensors()
in_detail = interp.get_input_details()[0]
out_detail = interp.get_output_details()[0]
in_scale, in_zp = in_detail["quantization"]
out_scale, out_zp = out_detail["quantization"]

eval_idx = np.random.default_rng(1).choice(len(x_test_f), EVAL_N, replace=False)
correct = 0
for i in eval_idx:
    img_int8 = np.round(x_test_f[i:i+1] / in_scale + in_zp).astype(np.int8)
    interp.set_tensor(in_detail["index"], img_int8)
    interp.invoke()
    out = interp.get_tensor(out_detail["index"])[0]
    if int(np.argmax(out)) == int(y_test[i]):
        correct += 1
INT8_ACC = 100.0 * correct / EVAL_N
EVAL_N_ACTUAL = EVAL_N
print(f"INT8 test accuracy: {INT8_ACC:.2f}%  ({correct}/{EVAL_N})")
print(f"input:  scale={in_scale}  zero_point={in_zp}")
print(f"output: scale={out_scale}  zero_point={out_zp}")

Saved artifact at '/tmp/tmpc7vxa05e'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name='keras_tensor_196')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  139932728501904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139932806166864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139932728505552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139932728504400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139932728495184: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139932728505936: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Wrote: mnist_int8.tflite  (41736 bytes)


/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


INT8 test accuracy: 99.35%  (1987/2000)
input:  scale=0.003921568859368563  zero_point=-128
output: scale=0.00390625  zero_point=-128


## 5. 分析 tflite、ops

用官方內建的 Analyzer，看這個模型實際用了哪些 op -- 等一下寫 `MicroMutableOpResolver<N>` 只需要註冊這裡列出來的。

In [ ]:
tf.lite.experimental.Analyzer.analyze(model_content=tflite_model)

=== TFLite ModelAnalyzer ===

Your TFLite model has '1' subgraph(s). In the subgraph description below,
T# represents the Tensor numbers. For example, in Subgraph#0, the CONV_2D op takes
tensor #0 and tensor #9 and tensor #8 as input and produces tensor #10 as output.

Subgraph#0 main(T#0) -> [T#19]
  Op#0 CONV_2D(T#0, T#9, T#8[-2242, -289, -211, -16351, 142, ...]) -> [T#10]
  Op#1 MAX_POOL_2D(T#10) -> [T#11]
  Op#2 CONV_2D(T#11, T#7, T#6[-3623, -1724, -1597, 469, -1330, ...]) -> [T#12]
  Op#3 MAX_POOL_2D(T#12) -> [T#13]
  Op#4 SHAPE(T#13) -> [T#14]
  Op#5 STRIDED_SLICE(T#14, T#1[0], T#2[1], T#2[1]) -> [T#15]
  Op#6 PACK(T#15, T#3[1600]) -> [T#16]
  Op#7 RESHAPE(T#13, T#16) -> [T#17]
  Op#8 FULLY_CONNECTED(T#17, T#5, T#4[-654, 2087, -352, -627, -1358, ...]) -> [T#18]
  Op#9 SOFTMAX(T#18) -> [T#19]

Tensors of Subgraph#0
  T#0(serving_default_keras_tensor_196:0) shape_signature:[-1, 28, 28, 1], type:INT8
  T#1(arith.constant) shape:[1], type:INT32 RO 4 bytes, buffer: 2, data:[0]
  T#2(a

## 6. 轉換 model -- 產生 header/cc

跟 Stanley 的做法一樣：不用 `xxd -i`，自己寫 template，順便把 scale/zero_point/accuracy 當註解寫進去，方便之後追溯這顆模型的來源。

In [ ]:
def format_byte_array(data: bytes, indent="    ") -> str:
    lines = []
    for i in range(0, len(data), 12):
        chunk = data[i:i + 12]
        lines.append(indent + ", ".join(f"0x{b:02x}" for b in chunk) + ",")
    return "\n".join(lines)

HEADER_TEMPLATE = '''#pragma once
#include <stdint.h>

/* MNIST digit (0-9) classifier, int8 quantized.
 * Input:  28x28x1 int8, scale={in_scale}, zero_point={in_zp}
 *         (pixel/255.0, then quantized)
 * Output: 10 classes (digit 0-9), int8, scale={out_scale}, zero_point={out_zp}
 * Size: {size} bytes ({size_kb:.1f} KB)
 * float accuracy: {float_acc:.2f}%   int8 accuracy: {int8_acc:.2f}% (n={eval_n})
 *
 * Usage:
 *   #include "mnist_model_data.h"
 *   const tflite::Model* model = tflite::GetModel(g_mnist_model_data);
 */
extern const uint8_t g_mnist_model_data[];
extern const uint32_t g_mnist_model_data_len;
'''

CC_HEADER_TEMPLATE = '''#include "mnist_model_data.h"

/* Generated by mnist_train_colab.ipynb
 * epochs={epochs}  float_acc={float_acc:.2f}%  int8_acc={int8_acc:.2f}% (n={eval_n})  size={size} bytes
 */
alignas(8) const uint8_t g_mnist_model_data[] = {{
'''

CC_FOOTER = '''};
const uint32_t g_mnist_model_data_len = sizeof(g_mnist_model_data);
'''

H_PATH = "mnist_model_data.h"
CC_PATH = "mnist_model_data.cc"

header = HEADER_TEMPLATE.format(
    in_scale=in_scale, in_zp=in_zp, out_scale=out_scale, out_zp=out_zp,
    size=len(tflite_model), size_kb=len(tflite_model) / 1024.0,
    float_acc=FLOAT_ACC, int8_acc=INT8_ACC, eval_n=EVAL_N_ACTUAL,
)
with open(H_PATH, "w", encoding="utf-8") as f:
    f.write(header)
print(f"Wrote: {H_PATH}")

cc = CC_HEADER_TEMPLATE.format(
    epochs=EPOCHS, float_acc=FLOAT_ACC, int8_acc=INT8_ACC,
    eval_n=EVAL_N_ACTUAL, size=len(tflite_model),
)
cc += format_byte_array(tflite_model)
cc += "\n" + CC_FOOTER
with open(CC_PATH, "w", encoding="utf-8") as f:
    f.write(cc)
print(f"Wrote: {CC_PATH}  ({len(tflite_model)} bytes)")

Wrote: mnist_model_data.h
Wrote: mnist_model_data.cc  (41736 bytes)


## 8. AMB82-Mini 專用匯出 — CNN-RGB + MobileNetV2 backbone

背景：AMB82 線上轉換工具的 **CNN-GRAY** 路徑目前伺服器端有 bug（`inputmeta.yml` 產生的
`preproc_type` 是壞掉的模板值，YAML 都 parse 不過），回報已送出但要等 Realtek 修，不可控。

**CNN-RGB** 路徑試過小的自訂 CNN 也會炸（`cannot reshape array of size 1 into shape (1,1280)`），
懷疑後端 import/quantize pipeline 是針對 MobileNetV2（bottleneck 剛好 1280 維）寫的 shape
inference，非 MobileNetV2 架構的模型對不上就崩潰。同一封回信也提到：

> Pro2 RGB w/h value from maximum to minimum is 1280x704 to 96x96. The value is recommended to be the multiples of 32.

所以這次改用真的 `tf.keras.applications.MobileNetV2`（`include_top=False, pooling="avg"`，輸出
剛好是 1280 維）當 backbone，輸入尺寸放大到 96x96（下限、32 的倍數）、灰階複製成三通道。
這是目前唯一還能自己驗證、不用等審核的線上路徑。

`weights=None` 從頭訓練，先求能不能通過官方的 import/quantize/export 三步驟；如果能通過，
之後要拉高準確率再考慮 `weights="imagenet"` fine-tune。

In [ ]:
import zipfile
import os
from PIL import Image

IMG_SIZE = 96  # Pro2 RGB 下限，且是 32 的倍數

def resize_repeat(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))  # (28,28,1) float32 -> (96,96,1)
    img = tf.image.grayscale_to_rgb(img)               # (96,96,1) -> (96,96,3)
    return img, label

# 切一份 validation（跟 cell-8 用同樣的 10% 比例、固定 seed，方便對照）
rng = np.random.default_rng(42)
perm = rng.permutation(len(x_train_f))
n_val = int(len(x_train_f) * 0.1)
val_idx, tr_idx = perm[:n_val], perm[n_val:]

BATCH_SIZE_RGB = 64  # 96x96x3 float32 比 28x28x1 重很多，batch 縮小一點

def make_ds(x, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(10000, seed=42)
    return (ds.map(resize_repeat, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(BATCH_SIZE_RGB)
              .prefetch(tf.data.AUTOTUNE))

train_ds = make_ds(x_train_f[tr_idx], y_train[tr_idx], shuffle=True)
val_ds = make_ds(x_train_f[val_idx], y_train[val_idx])
test_ds = make_ds(x_test_f, y_test)

def build_model_mobilenetv2():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights=None,     # 先求 pipeline 能過；能過再考慮 weights="imagenet" fine-tune 拉準確率
        pooling="avg",    # GlobalAveragePooling -> 剛好對齊官方 pipeline 預期的 1280 維
    )
    x = backbone(inputs)
    x = tf.keras.layers.Dropout(0.5)(x)
    x = tf.keras.layers.Dense(NUM_CLASSES)(x)
    outputs = tf.keras.layers.Softmax()(x)
    return tf.keras.Model(inputs, outputs)

EPOCHS_MBV2 = 6  # 這次目的是驗證轉換 pipeline，不是衝準確率，先跑少一點

tf.random.set_seed(42)
mbv2_model = build_model_mobilenetv2()
mbv2_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
mbv2_history = mbv2_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_MBV2, verbose=2)

mbv2_test_loss, mbv2_test_acc = mbv2_model.evaluate(test_ds, verbose=0)
print(f"MobileNetV2-RGB float test accuracy: {mbv2_test_acc*100:.2f}%  (loss={mbv2_test_loss:.4f})")

# --- 存 .h5，關掉 optimizer（官方文件要求） ---
AMB82_H5_PATH = "mnist_amb82_mbv2.h5"
mbv2_model.save(AMB82_H5_PATH, include_optimizer=False)

AMB82_H5_ZIP = "mnist_amb82_mbv2.zip"
with zipfile.ZipFile(AMB82_H5_ZIP, "w") as zf:
    zf.write(AMB82_H5_PATH)
print(f"Wrote: {AMB82_H5_ZIP}")

# --- 校準圖片：CNN-RGB 路徑吃圖片（不是 tensor），96x96 三通道 jpg，每個數字一張 ---
CALIB_DIR = "calib_images_mbv2"
os.makedirs(CALIB_DIR, exist_ok=True)
calib_paths = []
for digit in range(10):
    idx = int(np.where(y_train == digit)[0][0])
    img28 = x_train_f[idx]                                          # (28,28,1) float32 [0,1]
    img96 = tf.image.resize(img28[None], (IMG_SIZE, IMG_SIZE))[0]     # (96,96,1)
    img96 = tf.image.grayscale_to_rgb(img96).numpy()                 # (96,96,3)
    img_uint8 = np.clip(img96 * 255.0, 0, 255).astype(np.uint8)
    p = os.path.join(CALIB_DIR, f"digit_{digit}.jpg")
    Image.fromarray(img_uint8).save(p)
    calib_paths.append(p)

CALIB_ZIP = "calib_images_mbv2.zip"
with zipfile.ZipFile(CALIB_ZIP, "w") as zf:
    for p in calib_paths:
        zf.write(p)
print(f"Wrote: {CALIB_ZIP}  ({len(calib_paths)} .jpg files)")


下載 `.zip`（裡面是 `.h5`）跟校準用的 jpg 圖片，去 `amebaiot.com` 的 AI Model Conversion 頁面上傳。表單欄位對照：

- Model: **CNN-RGB**（不是 CNN-GRAY，GRAY 路徑目前伺服器 bug，見上方說明）
- Quantize Type: **UINT8**
- reverse_channel: **false**
- scale: **0.00392156** (= 1/255，跟訓練時 pixel/255.0 的正規化一致)
- h5 upload: `mnist_amb82_mbv2.zip`
- 校準圖片 upload: `calib_images_mbv2.zip`（10 張 96x96 RGB jpg，不是 tensor —— RGB 路徑吃圖片）

**這次上傳的目的只是驗證能不能通過 import/quantize/export**，準確率（目前只跑 6 epoch、weights=None）
還沒調到位。如果卡在同一個 `reshape...(1,1280)` 的錯誤，代表這條線上路徑走到底了，不用再試，
直接等離線工具核准；如果通過了，回來把 `EPOCHS_MBV2` 拉高、或改 `weights="imagenet"` 做
fine-tune 再重新匯出一次正式版本。


In [ ]:
save_output(AMB82_H5_ZIP)
save_output(CALIB_ZIP)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. AMB82 專用匯出 v2 — PyTorch MobileNetV2（官方唯一記錄過會過的組合）

Keras 版（上面 cell 16）兩次都死在同一個 `(1,1280)` reshape：一次是完全跟 1280 無關的小 CNN
（自己的 reshape 目標是 1600），一次是真的 Keras MobileNetV2（bottleneck 本來就是 1280）。
兩次錯誤位置、錯誤訊息逐字相同，代表這個 reshape 節點**根本不是我們模型裡的東西**，是
後端幫 CNN-RGB 上傳都套用的固定模板，而這個模板顯然是照 PyTorch 匯出的 ONNX graph 設計的：
Keras 的 `tf.keras.applications.MobileNetV2` 是「model 包 model」的巢狀結構，匯出 ONNX 後模板
抓不到預期的張量，shape inference 就退化成算出「1」。

論壇 [MobileNetV2 offline conversion](https://forum.amebaiot.com/t/mobilenetv2-offline-conversion/4849)
官方也明講：「目前 SDK 只支援 PyTorch 訓練的 MobileNetV2」，Keras 版本會報
`Functional`/`Rescaling` 不支援。所以這次整段換 PyTorch：

- 用 `torchvision.models.mobilenet_v2`（單一扁平的 `nn.Module`，沒有巢狀 sub-model 的問題）
- 直接 `torch.onnx.export()`，**跳過 `.h5`**，也就跳過 Realtek 後端自己那個有問題的 h5→onnx 轉換
- 匯出時**不設 `dynamic_axes`**，固定 batch=1，整張圖沒有任何動態維度，避免對方那個
  以 numpy 為主的 shape-inference 把不確定維度誤判成 1
- `weights=None` 從頭訓練，一樣先求能不能通過 import/quantize/export，不是衝準確率

跟 Pico 2 那條線（cell 1-14）完全獨立，不會互相影響。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
import torchvision

print("torch:", torch.__version__)

IMG_SIZE_PT = 96  # 跟 Keras 版一樣，Pro2 RGB 下限、32 的倍數
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


class MNISTMobileNetV2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = torchvision.models.mobilenet_v2(weights=None)
        in_features = self.backbone.classifier[1].in_features  # 1280
        self.backbone.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)


class MNISTResizeDataset(torch.utils.data.Dataset):
    """把 28x28x1 [0,1] 的 MNIST 圖片即時 resize 成 96x96x3，不預先攤開整個資料集(避免爆記憶體)。"""

    def __init__(self, x, y):
        self.x = x  # (N,28,28,1) float32 numpy, [0,1]
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.x[idx]).permute(2, 0, 1)  # (1,28,28)
        img = Fnn.interpolate(img.unsqueeze(0), size=(IMG_SIZE_PT, IMG_SIZE_PT),
                               mode="bilinear", align_corners=False).squeeze(0)
        img = img.repeat(3, 1, 1)  # (3,96,96)
        return img, int(self.y[idx])


# 沿用跟 cell-16 一樣的 train/val 切分 (同 seed，方便對照)
train_ds_pt = MNISTResizeDataset(x_train_f[tr_idx], y_train[tr_idx])
val_ds_pt = MNISTResizeDataset(x_train_f[val_idx], y_train[val_idx])
test_ds_pt = MNISTResizeDataset(x_test_f, y_test)

BATCH_SIZE_PT = 64
train_loader = torch.utils.data.DataLoader(train_ds_pt, batch_size=BATCH_SIZE_PT, shuffle=True, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_ds_pt, batch_size=BATCH_SIZE_PT, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_ds_pt, batch_size=BATCH_SIZE_PT, num_workers=2)

torch.manual_seed(42)
pt_model = MNISTMobileNetV2(NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(pt_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

EPOCHS_PT = 6  # 目的是驗證轉換 pipeline，不是衝準確率，先跑少一點

for epoch in range(EPOCHS_PT):
    pt_model.train()
    total, correct, running_loss = 0, 0, 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = pt_model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    train_acc = 100.0 * correct / total

    pt_model.eval()
    v_total, v_correct = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = pt_model(imgs)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total += imgs.size(0)
    val_acc = 100.0 * v_correct / v_total
    print(f"epoch {epoch+1}/{EPOCHS_PT}  loss={running_loss/total:.4f}  "
          f"train_acc={train_acc:.2f}%  val_acc={val_acc:.2f}%")

pt_model.eval()
t_total, t_correct = 0, 0
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = pt_model(imgs)
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total += imgs.size(0)
PT_TEST_ACC = 100.0 * t_correct / t_total
print(f"PyTorch MobileNetV2 test accuracy: {PT_TEST_ACC:.2f}%")

In [ ]:
pt_model_cpu = pt_model.to("cpu").eval()
dummy_input = torch.zeros(1, 3, IMG_SIZE_PT, IMG_SIZE_PT)

ONNX_PT_PATH = "mnist_amb82_mbv2_pt.onnx"
# torch >= 2.9 的 torch.onnx.export 預設 dynamo=True，要裝 onnxscript。
# 但我們要的是舊的 TorchScript 匯出器（opset 11 這組參數就是給它寫的，
# Realtek acuity 5.21.1 也只看得懂那種圖）-> 明確關掉 dynamo。
_onnx_kwargs = dict(
    input_names=["input"],
    output_names=["output"],
    opset_version=11,       # 刻意用低 opset，跟 acuity-toolkit-rel-5.21.1 的相容性比較好
    do_constant_folding=True,
    # 不給 dynamic_axes -> 固定 batch=1，轉進去就沒有任何動態維度
)
try:
    torch.onnx.export(pt_model_cpu, dummy_input, ONNX_PT_PATH,
                      dynamo=False, **_onnx_kwargs)
except TypeError:
    # 舊版 torch 沒有 dynamo 這個參數
    torch.onnx.export(pt_model_cpu, dummy_input, ONNX_PT_PATH, **_onnx_kwargs)
print(f"Wrote: {ONNX_PT_PATH}")

ONNX_PT_ZIP = "mnist_amb82_mbv2_pt.zip"
with zipfile.ZipFile(ONNX_PT_ZIP, "w") as zf:
    zf.write(ONNX_PT_PATH)
print(f"Wrote: {ONNX_PT_ZIP}")

# --- 校準圖片：一樣是 96x96 RGB jpg，每個數字一張 ---
CALIB_DIR_PT = "calib_images_mbv2_pt"
os.makedirs(CALIB_DIR_PT, exist_ok=True)
calib_paths_pt = []
for digit in range(10):
    idx = int(np.where(y_train == digit)[0][0])
    img, _ = MNISTResizeDataset(x_train_f, y_train)[idx]   # (3,96,96) torch float32 [0,1]
    img_np = (img.permute(1, 2, 0).numpy() * 255.0)
    img_np = np.clip(img_np, 0, 255).astype(np.uint8)       # (96,96,3)
    p = os.path.join(CALIB_DIR_PT, f"digit_{digit}.jpg")
    Image.fromarray(img_np).save(p)
    calib_paths_pt.append(p)

CALIB_ZIP_PT = "calib_images_mbv2_pt.zip"
with zipfile.ZipFile(CALIB_ZIP_PT, "w") as zf:
    for p in calib_paths_pt:
        zf.write(p)
print(f"Wrote: {CALIB_ZIP_PT}  ({len(calib_paths_pt)} .jpg files)")

save_output(ONNX_PT_ZIP)
save_output(CALIB_ZIP_PT)


下載 `mnist_amb82_mbv2_pt.zip`（裡面是 `.onnx`，這次**直接上傳 onnx，不要 h5**）跟
`calib_images_mbv2_pt.zip`，去 `amebaiot.com` 的 AI Model Conversion 頁面上傳。表單欄位對照：

- Model: **CNN-RGB**
- Quantize Type: **UINT8**
- reverse_channel: **false**
- scale: **0.00392156** (= 1/255，PyTorch 這邊訓練時也是餵 [0,1] 沒有另外用 ImageNet mean/std)
- onnx upload: `mnist_amb82_mbv2_pt.zip`
- 校準圖片 upload: `calib_images_mbv2_pt.zip`

**這是目前手上最後一個、也是唯一有官方留言佐證「PyTorch 訓練的 MobileNetV2 可以過」的組合。**
如果這次還是卡在同一個 `reshape...(1,1280)`，就代表線上工具這條路真的走到底了——不是你架構選錯、
不是前處理設錯，兩個框架、三種完全不同的架構都死在同一行，證據已經足夠強，這時候就把這三次
（原始小 CNN、Keras MobileNetV2、PyTorch MobileNetV2）的 log 一起送給 Realtek，然後專心等離線
工具核准，不用再繼續猜下去。


## 7. 下載

把 `.h` / `.cc` 抓回你的 Mac，之後我們接進 `pico2-tflm-hello` 專案。

In [ ]:
save_output(H_PATH)
save_output(CC_PATH)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10. 跑分模型:AMB82-MINI vs Pico 2(這一節才是跑分用的)

**為什麼不能直接拿 cell 6 那個 28x28 模型上 AMB82:**
線上轉檔器只有兩條路。CNN-GRAY 目前是壞的(見第 5 節);CNN-RGB 的輸入尺寸
Realtek 官方說「1280x704 到 96x96,建議 32 的倍數」。28x28 不在範圍內 ->
那個模型沒有任何合法的上傳形狀。

**所以跑分模型定義成 96x96x3 輸入的小 CNN。**
不用 MobileNetV2(太大,而且會撞第 5/6 節那個 `(1,1280)` reshape bug)。
第一層直接 stride=2 把解析度砍半,最大中間張量約 47x47x8 = 17.6 KB,
Pico 2(RP2350,520 KB SRAM)吃得下,AMB82 的 VIP NPU 更不是問題。

**公平性**:兩邊跑同一組權重、同一個圖。
- AMB82:`.h5` -> 線上轉 `.nb` -> 燒 flash,量 callback 之間的 `micros()` 間隔
- Pico 2:同一個 Keras model -> int8 `.tflite` -> C array,量 `interpreter.Invoke()` 前後

**注意:這個模型跟第 2 節的 28x28 模型不是同一個**,所以 int8 精度會不一樣,
但跑分要的是「同一個圖在兩顆晶片上的時間」,精度只要別爛掉就行。

**AMB82 端兩個地雷**(寫 sketch 時記得):
- `configInputImageColor(0)` 會呼叫 `img_rgb2gray()`,雖然那個巨集叫 `IMAGERGB`。這模型吃 RGB,要傳 1。
- 官方 example 的 `VideoSetting configNN(..., 10, VIDEO_RGB, 0)`,那個 10 是 **10 fps 上限**。
  不改的話量到的是節流後的數字,不是推論速度。跑分要往上調到飽和為止。


In [ ]:
import zipfile
from PIL import Image

BENCH_SIZE = 96          # Pro2 CNN-RGB 下限,且是 32 的倍數
BENCH_EPOCHS = 8
BENCH_BATCH = 64


def build_bench_model():
    # 96x96x3 小 CNN。注意:刻意不放任何 augmentation / Rescaling 層 --
    # 那些層會被烘進存出的圖裡,轉檔器不認得。要 augmentation 就在 tf.data 裡做。
    # 不用 tf.keras.Input() 當 Sequential 第一個元素 -- Keras 3 可以,
    # Keras 2 不行(Input() 回傳 KerasTensor 不是 Layer)。改用 input_shape,兩邊通用。
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(8, 3, strides=2, activation="relu",
                               input_shape=(BENCH_SIZE, BENCH_SIZE, 3)),  # 47x47x8
        tf.keras.layers.MaxPooling2D(2),                              # 23x23x8
        tf.keras.layers.Conv2D(16, 3, activation="relu"),             # 21x21x16
        tf.keras.layers.MaxPooling2D(2),                              # 10x10x16
        tf.keras.layers.Conv2D(32, 3, activation="relu"),             # 8x8x32
        tf.keras.layers.MaxPooling2D(2),                              # 4x4x32
        tf.keras.layers.Reshape((4 * 4 * 32,)),
        tf.keras.layers.Dense(NUM_CLASSES),
        tf.keras.layers.Softmax(),
    ])


build_bench_model().summary()


def to_rgb96(img, label):
    img = tf.image.resize(img, (BENCH_SIZE, BENCH_SIZE))
    img = tf.image.grayscale_to_rgb(img)
    return img, label


def bench_ds(x, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if shuffle:
        ds = ds.shuffle(10000, seed=42)
    return (ds.map(to_rgb96, num_parallel_calls=tf.data.AUTOTUNE)
              .batch(BENCH_BATCH).prefetch(tf.data.AUTOTUNE))


_rng = np.random.default_rng(42)
_perm = _rng.permutation(len(x_train_f))
_nval = int(len(x_train_f) * 0.1)
_vi, _ti = _perm[:_nval], _perm[_nval:]

bench_train = bench_ds(x_train_f[_ti], y_train[_ti], shuffle=True)
bench_val = bench_ds(x_train_f[_vi], y_train[_vi])
bench_test = bench_ds(x_test_f, y_test)

tf.random.set_seed(42)
bench_model = build_bench_model()
bench_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
bench_model.fit(bench_train, validation_data=bench_val, epochs=BENCH_EPOCHS, verbose=2)
_bl, _ba = bench_model.evaluate(bench_test, verbose=0)
BENCH_FLOAT_ACC = _ba * 100
print("bench model FLOAT test accuracy: %.2f%%" % BENCH_FLOAT_ACC)


In [ ]:
# --- AMB82 端:.h5 + zip + 校準圖 ---
BENCH_H5 = "mnist_bench96.h5"
try:
    # include_optimizer=False 是官方硬性要求
    bench_model.save(BENCH_H5, include_optimizer=False)
except Exception as e:
    print("h5 存檔失敗 (" + str(e) + ") -> 改存 .keras")
    BENCH_H5 = "mnist_bench96.keras"
    bench_model.save(BENCH_H5)
    print("!! 轉檔器要的是 .h5。若走到這裡,請回第 0 節開 USE_LEGACY_KERAS 重跑")
print("saved:", BENCH_H5, os.path.getsize(BENCH_H5), "bytes")

BENCH_H5_ZIP = "mnist_bench96.zip"
with zipfile.ZipFile(BENCH_H5_ZIP, "w") as zf:
    zf.write(BENCH_H5)
print("Wrote: " + BENCH_H5_ZIP)

BENCH_CALIB_DIR = "calib_bench96"
os.makedirs(BENCH_CALIB_DIR, exist_ok=True)
_paths = []
for digit in range(10):
    i = int(np.where(y_train == digit)[0][0])
    im = tf.image.resize(x_train_f[i][None], (BENCH_SIZE, BENCH_SIZE))[0]
    im = tf.image.grayscale_to_rgb(im).numpy()
    p = os.path.join(BENCH_CALIB_DIR, "digit_%d.jpg" % digit)
    Image.fromarray(np.clip(im * 255.0, 0, 255).astype(np.uint8)).save(p)
    _paths.append(p)

BENCH_CALIB_ZIP = "calib_bench96.zip"
with zipfile.ZipFile(BENCH_CALIB_ZIP, "w") as zf:
    for p in _paths:
        zf.write(p)
print("Wrote: " + BENCH_CALIB_ZIP + " (%d jpg)" % len(_paths))

save_output(BENCH_H5_ZIP)
save_output(BENCH_CALIB_ZIP)


### 10b. 上傳到 amebaiot.com AI Model Conversion

- Model: **CNN-RGB**
- Quantize Type: **UINT8**
- reverse_channel: **false**
- scale: **0.00392156**  (= 1/255,和訓練時 pixel/255.0 一致)
- h5 upload: `mnist_bench96.zip`
- 校準圖 upload: `calib_bench96.zip`

轉出來的 `.nb` 改名成 `img_class_cnn.nb`,放進 sketch 資料夾,
Arduino IDE 選 `Tools -> NN Model Load From = flash`,sketch 裡 `modelSelect(..., CUSTOMIZED_IMGCLASS)`。
build 時 `cmodel_backup` 會把它換進 `variants/common_nn_models/`,原檔備份成 `Dbackup_img_class_cnn.nb`。
**不需要 SD 卡。**

如果這裡又出現 `cannot reshape array of size 1 into shape (1,1280)`,
那就證明線上管線對「非 MobileNetV2 拓樸」根本無效,只剩離線 Acuity toolkit 一條路(要申請)。


In [ ]:
# --- Pico 2 端:同一組權重 -> int8 tflite -> C array ---
BENCH_TFLITE = "mnist_bench96_int8.tflite"


def bench_representative():
    rng = np.random.default_rng(0)
    for i in rng.choice(len(x_train_f), 200, replace=False):
        im = tf.image.resize(x_train_f[i:i + 1], (BENCH_SIZE, BENCH_SIZE))
        im = tf.image.grayscale_to_rgb(im)
        yield [tf.cast(im, tf.float32)]


conv = tf.lite.TFLiteConverter.from_keras_model(bench_model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = bench_representative
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conv.inference_input_type = tf.int8
conv.inference_output_type = tf.int8
bench_tflite = conv.convert()
with open(BENCH_TFLITE, "wb") as f:
    f.write(bench_tflite)
print("Wrote: %s  (%d bytes)" % (BENCH_TFLITE, len(bench_tflite)))

_it = tf.lite.Interpreter(model_content=bench_tflite)
_it.allocate_tensors()
_id, _od = _it.get_input_details()[0], _it.get_output_details()[0]
print("input:", _id["shape"], _id["dtype"], "scale/zp:", _id["quantization"])
print("output:", _od["shape"], _od["dtype"], "scale/zp:", _od["quantization"])

BENCH_H = "mnist_bench96_model_data.h"
BENCH_CC = "mnist_bench96_model_data.cc"
NL = chr(10)
with open(BENCH_H, "w") as f:
    f.write("#pragma once" + NL + "#include <stdint.h>" + NL
            + "/* 96x96x3 int8 MNIST bench model, %d bytes, float acc %.2f%% */"
              % (len(bench_tflite), BENCH_FLOAT_ACC) + NL
            + "extern const uint8_t g_mnist_bench96_model_data[];" + NL
            + "extern const uint32_t g_mnist_bench96_model_data_len;" + NL)

_rows = []
for i in range(0, len(bench_tflite), 12):
    _rows.append("    " + ", ".join("0x%02x" % b for b in bench_tflite[i:i + 12]) + ",")
with open(BENCH_CC, "w") as f:
    f.write('#include "' + BENCH_H + '"' + NL + NL
            + "alignas(8) const uint8_t g_mnist_bench96_model_data[] = {" + NL
            + NL.join(_rows) + NL + "};" + NL
            + "const uint32_t g_mnist_bench96_model_data_len = %d;" % len(bench_tflite) + NL)
print("Wrote: " + BENCH_H + " / " + BENCH_CC)

save_output(BENCH_H)
save_output(BENCH_CC)
save_output(BENCH_TFLITE)


## 11. 打包下載(Kaggle 用)

Kaggle 沒有 `files.download()`,不會自動下載。跑完這格會:
1. 把所有產物打成一個 `amb82_bench_bundle.zip`
2. 列出 `/kaggle/working` 的完整內容 -- **先看這份清單確認東西都在**,再去下載

下載方式:右側 `Output` 面板點檔名;或右上 `Save Version -> Save & Run All`,
跑完進那個 version 的 `Output` 分頁按 `Download All`。


In [ ]:
import zipfile, os, glob

WANT = [
    "mnist_bench96.zip", "calib_bench96.zip",                           # -> AMB82 線上轉檔
    "mnist_bench96_model_data.h", "mnist_bench96_model_data.cc",
    "mnist_bench96_int8.tflite",                                        # -> Pico 2 (跑分)
    "mnist_int8.tflite", "mnist_model_data.h", "mnist_model_data.cc",   # -> Pico 2 (舊 28x28)
    "mnist_tiny.h5",
]

BUNDLE = os.path.join(OUT_DIR, "amb82_bench_bundle.zip")
missing = []
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in WANT:
        for p in (name, os.path.join(OUT_DIR, name)):
            if os.path.exists(p):
                zf.write(p, os.path.basename(p))
                break
        else:
            missing.append(name)

print("bundle:", BUNDLE, os.path.getsize(BUNDLE), "bytes")
with zipfile.ZipFile(BUNDLE) as zf:
    for n in zf.namelist():
        print("   +", n)
for n in missing:
    print("   (缺)", n)

print()
print("cwd:", os.getcwd())
print("/kaggle/working 全部內容:")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    print("   %-42s %9d" % (os.path.basename(p), os.path.getsize(p)))

if missing:
    print()
    print("有缺檔 -> 代表對應那節沒跑到(Run All 撞到 exception 就會中斷)。")
    print("第 8/9 節的 MobileNetV2 請跳過:CPU 要幾小時,而且是已知失敗的路線。")


## 12. CNN-GRAY 重現用(給 Realtek 的 bug report)

這一節的目的**不是**做出可用的模型,而是產出一組**乾淨、最小、可重現**的
CNN-GRAY 上傳素材,用來回報線上轉檔器的 GRAY 路線壞掉。

刻意做兩個尺寸,兩者架構完全相同、只差輸入解析度:

| 檔案 | 輸入 | 為什麼要這個 |
|---|---|---|
| `mnist_gray96.zip` | 96x96x1 | 96 是 32 的倍數,踩在官方說的 RGB 下限上 -> 尺寸不能當理由 |
| `mnist_gray28.zip` | 28x28x1 | MNIST 原生尺寸,最小、最單純的案例 |

如果**兩個都以相同方式失敗**,就排除了「輸入尺寸不支援」這個解釋,
問題只能在 GRAY 這條 code path 本身。

模型刻意保持極簡:兩層 Conv + 固定 Reshape + Dense,**沒有** global average pooling、
**沒有** augmentation / Rescaling 層、**沒有**任何動態 shape。
(依 CNN-RGB 的 log 診斷,動態 Reshape 是 acuity shape inference 掛掉的原因,
這裡連 Flatten / global pooling 都不用，改用算好常數的 Reshape,才能確定失敗是 GRAY path 造成的、而不是拓樸問題。)

上傳設定:

- Model: **CNN-GRAY**
- Quantize Type: **UINT8**
- scale: **0.00392156** (= 1/255)
- 校準圖:單通道 8-bit 灰階 jpg(PIL mode "L")


In [ ]:
import zipfile, os
from PIL import Image


def gray_dims(size):
    """把每一層的空間尺寸算出來,好讓 Reshape 用固定常數(不產生動態 shape)。"""
    h = (size - 3) // 2 + 1     # Conv2D(8, 3, strides=2)
    h = h // 2                  # MaxPooling2D(2)
    h = h - 2                   # Conv2D(16, 3)
    h = h // 2                  # MaxPooling2D(2)
    assert h >= 1, "size=%d 太小,卷積堆疊會塌成 0" % size
    return h, h * h * 16


def build_gray_model(size):
    """單通道極簡 CNN。
    刻意不用 Input() 當 Sequential 第一層(Keras 2 不接受);
    刻意**不用** GlobalAveragePooling/GlobalMaxPooling 也不用 Flatten,
    改用算好常數的 Reshape —— global pooling 與 Flatten 在 tf2onnx 底下
    常會產生 Shape->Gather->Concat->Reshape 的動態子圖,
    那正是 CNN-RGB log 裡 acuity shape inference 掛掉的結構。
    這裡整條路徑都是固定尺寸,失敗就只能歸因於 GRAY code path。"""
    _, flat = gray_dims(size)
    return tf.keras.Sequential([
        tf.keras.layers.Conv2D(8, 3, strides=2, activation="relu",
                               input_shape=(size, size, 1)),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Conv2D(16, 3, activation="relu"),
        tf.keras.layers.MaxPooling2D(2),
        tf.keras.layers.Reshape((flat,)),       # 常數 target shape,無動態 shape
        tf.keras.layers.Dense(NUM_CLASSES),
        tf.keras.layers.Softmax(),
    ])


def make_gray_bundle(size, epochs=6, batch=128):
    """訓練 -> 存 .h5 -> zip -> 產 10 張單通道校準 jpg -> zip。回傳 (h5_zip, calib_zip, acc)。"""
    tag = "gray%d" % size

    if size == 28:
        xtr, xte = x_train_f, x_test_f
    else:
        xtr = tf.image.resize(x_train_f, (size, size)).numpy()
        xte = tf.image.resize(x_test_f, (size, size)).numpy()

    tf.random.set_seed(42)
    m = build_gray_model(size)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    m.fit(xtr, y_train, epochs=epochs, batch_size=batch, validation_split=0.1, verbose=2)
    _, acc = m.evaluate(xte, y_test, verbose=0)
    print("[%s] float test accuracy: %.2f%%" % (tag, acc * 100))
    m.summary()

    h5 = "mnist_%s.h5" % tag
    try:
        m.save(h5, include_optimizer=False)
    except (ValueError, TypeError):
        m.save(h5)
    h5zip = "mnist_%s.zip" % tag
    with zipfile.ZipFile(h5zip, "w") as zf:
        zf.write(h5)
    print("Wrote:", h5zip, os.path.getsize(h5zip), "bytes")

    # 校準圖:單通道 8-bit 灰階 jpg(PIL mode "L"),每個數字一張
    cdir = "calib_%s" % tag
    os.makedirs(cdir, exist_ok=True)
    paths = []
    for d in range(10):
        i = int(np.where(y_train == d)[0][0])
        im = xtr[i][:, :, 0]                                   # (size,size) float [0,1]
        u8 = np.clip(im * 255.0, 0, 255).astype(np.uint8)
        p = os.path.join(cdir, "digit_%d.jpg" % d)
        Image.fromarray(u8, mode="L").save(p)                  # mode="L" = 單通道灰階
        paths.append(p)
    czip = "calib_%s.zip" % tag
    with zipfile.ZipFile(czip, "w") as zf:
        for p in paths:
            zf.write(p)
    print("Wrote:", czip, "(%d jpg, mode=L)" % len(paths))

    # 確認存出的 jpg 真的是單通道
    with Image.open(paths[0]) as chk:
        print("  校準圖檢查:", chk.size, "mode=" + chk.mode,
              "-> " + ("單通道 OK" if chk.mode == "L" else "!! 不是單通道,GRAY 會對不上"))

    return h5zip, czip, acc * 100


GRAY_RESULTS = {}
for _sz in (96, 28):
    GRAY_RESULTS[_sz] = make_gray_bundle(_sz)

print()
print("=== 給 Realtek 的上傳素材 ===")
for _sz, (_h, _c, _a) in GRAY_RESULTS.items():
    print("  %dx%dx1  acc=%.2f%%  h5=%s  calib=%s" % (_sz, _sz, _a, _h, _c))
    save_output(_h)
    save_output(_c)
